# Analysis of Phylon and Strain Synteny

## NOTE: Currently only implemented for Panaroo Pangeneomes

In [ ]:
import pandas as pd
import numpy as np
import os

from pyphylon.util import load_config

from pyphylon.biointerp import generate_strain_vectors

import networkx as nx

#uncomment when done developing package for reloading
# from pyphylon.synteny import *

In [ ]:
# TEMP code for reloading module during development

import pyphylon.synteny

import importlib
importlib.reload(pyphylon.synteny)

from pyphylon.synteny import *

In [ ]:
CONFIG = load_config("config.yml")
WORKDIR = CONFIG["WORKDIR"]
SPECIES = CONFIG["PG_NAME"]
PANAROO = CONFIG['PANAROO']

# Load Data

In [ ]:
DF_GENES = os.path.join(WORKDIR, f'processed/CAR_genomes/df_genes.pickle.gz')
ENRICHED_METADATA = os.path.join(WORKDIR, 'interim/enriched_metadata_2d.csv')

DF_CORE_COMPLETE = os.path.join(WORKDIR, f'processed/CAR_genomes/df_core.csv')
DF_ACC_COMPLETE = os.path.join(WORKDIR, f'processed/CAR_genomes/df_acc.csv')
DF_RARE_COMPLETE = os.path.join(WORKDIR, f'processed/CAR_genomes/df_rare.csv')

In [ ]:
df_core_complete = pd.read_csv(DF_CORE_COMPLETE, index_col=0)
df_acc_complete = pd.read_csv(DF_ACC_COMPLETE, index_col=0)
df_rare_complete = pd.read_csv(DF_RARE_COMPLETE, index_col=0)

In [ ]:
# Load in (full) P matrix
df_genes = pd.read_pickle(DF_GENES)

In [ ]:
metadata = pd.read_csv(ENRICHED_METADATA, index_col=0, dtype='object')
metadata_complete = metadata[metadata.genome_status == 'Complete']

In [ ]:
# Filter P matrix for Complete sequences only
df_genes_complete = df_genes[metadata_complete.genome_id]
df_genes_complete = df_genes_complete.fillna(0) # replace N/A with 0
df_genes_complete = df_genes_complete.sparse.to_dense().astype('int8') # densify & typecast to int8 for space and compute reasons
inCompleteseqs = df_genes_complete.sum(axis=1) > 0 # filter for genes found in complete sequences
df_genes_complete = df_genes_complete[inCompleteseqs]

In [ ]:
# Load NMF Results
L_MATRIX = os.path.join(WORKDIR, f'processed/nmf-outputs/L_binarized.csv')
A_MATRIX = os.path.join(WORKDIR, f'processed/nmf-outputs/A_binarized.csv')

In [ ]:
L_binarized = pd.read_csv(L_MATRIX, index_col=0)
A_binarized = pd.read_csv(A_MATRIX, index_col=0)

# Generate Strain Information

In [ ]:
strain_vectors = generate_strain_vectors(WORKDIR, SPECIES, metadata_complete.genome_id.values, PANAROO = PANAROO)

# Create Graph Structure for Analysis

In [ ]:
phylon = 'phylon2'
phylon_genes = get_phylon_genes(L_binarized, phylon)

target_strains = get_phylon_strains(A_binarized, phylon)
strain_inputs = generate_strain_inputs(strain_vectors, metadata, target_strains)

In [ ]:
genes = get_gene_subset(
    core_genes=df_core_complete.index,
    phylon_genes=phylon_genes,
    mode="core_phylon",
)

G = build_pangenome_graph(
    strain_inputs,
    genes_to_keep=genes,
    circular = True
)

G = annotate_graph(G, 
                   core_genes=list(df_core_complete.index),
                   accessory_genes=list(df_acc_complete.index)
                  )

G  = annotate_support(G)

H = normalize_graph_support(
    G,
    len(target_strains),
)

In [ ]:
consensus_cycle = solve_max_weight_hamiltonian_cycle(
    H,
    time_limit=300000,
)

consensus_order = consensus_cycle[:-1]

In [ ]:
edge_scores = score_consensus(
    H,
    consensus_order,
)

edge_scores

In [ ]:
strain_scores_dict = {}

for strain,info in strain_inputs.items():

    s = chromosome_similarity(
        consensus_order,
        info["contigs"][0],
    )

    strain_scores_dict[strain] = s

df = pd.DataFrame(
    strain_scores_dict.items(),
    columns=["strain", "score"]
)

display(df)

# Sample Plots of the Data

In [ ]:
plot_edge_support(edge_scores)

In [ ]:
plot_presence_heatmap(
    consensus_order,
    strain_inputs,
)

In [ ]:
plot_accessory_insertions(
    consensus_order,
    strain_inputs,
)

In [ ]:
plot_strain_dotplot(
    consensus_order,
    strain_inputs[target_strains[0]]["contigs"][0],
    anchor_gene="dnaA",
)

In [ ]:
fig, ax = plot_consensus_synteny(
    consensus_order,
    strain_inputs,
)

In [ ]:
plot_backbone_comparison(
    consensus_order,
    strain_inputs[target_strains[1]]["contigs"][0],
    name1="Consensus",
    name2=strain,
    anchor_gene="dnaA"
)

# Comparison Between Two Phylon's Consensus Orders

In [ ]:
phylon2 = 'phylon1'
phylon_genes2 = get_phylon_genes(L_binarized, phylon2)

target_strains2 = get_phylon_strains(A_binarized, phylon2)
strain_inputs2 = generate_strain_inputs(strain_vectors, metadata, target_strains2)

In [ ]:
genes = get_gene_subset(
    core_genes=df_core_complete.index,
    phylon_genes=phylon_genes2,
    mode="core_phylon",
)

G = build_pangenome_graph(
    strain_inputs2,
    genes_to_keep=genes,
    circular = True
)

G = annotate_graph(G, 
                   core_genes=list(df_core_complete.index),
                   accessory_genes=list(df_acc_complete.index)
                  )

G  = annotate_support(G)

H = normalize_graph_support(
    G,
    len(target_strains2),
)

In [ ]:
consensus_cycle2 = solve_max_weight_hamiltonian_cycle(
    H,
    time_limit=300000,
)

consensus_order2 = consensus_cycle2[:-1]

In [ ]:
plot_backbone_comparison(
    consensus_order,
    consensus_order2,
    name1=phylon,
    name2=phylon2,
)